<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/04_lobbying_signal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Lobbying Signal

The third alternative-data signal, on the same `BaseSignal` contract as the congress
and government-contracts signals. It scores companies by their lobbying activity, and
it slots into the backtester and the composite identically to the others.

### Signal design

The thesis: a company stepping up its lobbying — especially an accelerating spend
across many policy issues — is investing to shape the rules that affect it, which can
precede favorable regulatory outcomes the market is slow to price. Four features carry
that idea:

| Feature | Hypothesis |
|---|---|
| `spend` | More recent lobbying dollars signal heavier policy engagement |
| `acceleration` | Spend rising above its prior run-rate is newly informative |
| `issue_breadth` | Engagement across many issues is broader than a single-issue push |
| `n_filings` | More filings indicate sustained intensity, not a one-off |

Filings key on the disclosure date, so the signal carries no look-ahead. Lobbying is
reported **quarterly**, so the lookback is longer than the event-driven signals
($180$ days) and acceleration is measured quarter-over-quarter.


## Setup

Defaults to mock data; set `USE_LIVE = True` for live Quiver lobbying data.


In [1]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True for live Quiver data

Working in: /content/quant-equity-research


## Install the package

This notebook's code lives in the repository under `src/quant_research/`; the clone above brought it in. The cell below installs the package in editable mode and puts it on the session path.

In [2]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.2.0


## Construct the signal

We compute as of today over a $180$-day lookback, treating the most recent $90$ days
as the window whose spend is compared against the prior quarter's run-rate.


In [3]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.lobbying import LobbyingSignal
import pandas as pd

client = QuiverClient(token=QUIVER_TOKEN) if USE_LIVE else QuiverClient(mock=True)
signal = LobbyingSignal(client, lookback_days=180, recent_days=90)

scores = signal.compute()
print(f"{len(scores)} tickers scored | data: {'live' if USE_LIVE else 'mock'}")

disp = scores.head(12).copy()
disp["spend_$M"] = (disp["spend"] / 1e6).round(2)
disp["accel_$M"] = (disp["acceleration"] / 1e6).round(2)
disp[["score", "spend_$M", "accel_$M", "issue_breadth", "n_filings"]]

1134 tickers scored | data: live


,score,spend_$M,accel_$M,issue_breadth,n_filings
ticker,,,,,
GM,100.0,16.01,8.19,18,37
META,90.8,15.74,0.47,26,49
AMZN,59.7,10.50,-0.35,19,28
GOOGL,58.9,8.74,-0.50,22,40
MO,58.3,8.73,-0.27,17,60
PFE,57.2,15.12,-5.04,12,32
GD,55.3,9.19,-0.35,16,44
ORCL,54.9,7.00,-0.16,22,45
AAPL,54.8,14.94,-8.26,19,26


### Reading the output

`score` is the $0$–$100$ conviction reading; the `*_n` columns are the cross-sectionally
normalized features behind it.


In [4]:
comp_cols = [c for c in scores.columns if c.endswith("_n")]
scores[["score"] + comp_cols].head(8)

,score,spend_n,acceleration_n,issue_breadth_n,n_filings_n
ticker,,,,,
GM,100.0,1.000,1.000,0.692,0.537
META,90.8,0.983,0.530,1.000,0.716
AMZN,59.7,0.656,0.481,0.731,0.403
GOOGL,58.9,0.546,0.471,0.846,0.582
MO,58.3,0.545,0.486,0.654,0.881
PFE,57.2,0.944,0.196,0.462,0.463
GD,55.3,0.574,0.481,0.615,0.642
ORCL,54.9,0.437,0.492,0.846,0.657


### Top names, shaded by acceleration

Shading by acceleration shows whether a high score reflects a large but steady lobbying
budget or a genuine recent step-up in spend.


In [5]:
import plotly.express as px

top = scores.head(15).reset_index()
fig = px.bar(top, x="score", y="ticker", orientation="h",
             color="acceleration_n", color_continuous_scale="Purples",
             labels={"acceleration_n": "acceleration"},
             title="Lobbying signal — top 15 (shading = acceleration)")
fig.update_layout(yaxis=dict(autorange="reversed"), height=460)
fig.show()

## Reading it honestly

- **Quarterly and lumpy.** Lobbying is filed once a quarter, so spend arrives in steps
  rather than a smooth flow; the longer lookback and quarter-over-quarter acceleration
  are built around that cadence.
- **Engagement, not outcome.** Heavy lobbying signals that a company is *trying* to
  shape policy, which is different from succeeding. Read the score as attention and
  positioning rather than a guaranteed regulatory win.
- **Direction is unlabeled.** The data shows spend and issues, not whether the company
  is lobbying for or against a given measure, so treat the signal as policy engagement
  in aggregate.

As with the others, this is a relative cross-sectional ranking, and the same `02`
engine validates it once the full set runs through together.


## Recap and next

Three of the four Hobbyist signals are now built on the shared contract: congress,
government contracts, and lobbying. One remains — the off-exchange signal, which plays
a confirming role and is shaped a little differently since it reads trading flow rather
than discrete events. After that comes the composite scorer that blends all four, a
combined backtest of the composite through the `02` engine, and then the dashboard.
